# GROUPING SETS, ROLLUP i CUBE — notatki referencyjne (SQL Server)

Przykłady na modelu: `fact_Sprzedaz` (ID_Klienta, ID_Placowki, DataSprzedazy, Kwota), `dim_Placowki` (ID_Placowki, Miasto, Region).

## 1. Problem, który rozwiązują — wiele poziomów agregacji w jednym zapytaniu

Klasyczne zadanie: potrzebujesz raportu sprzedaży z sumami na **kilku poziomach naraz** — per Region+Miasto, per sam Region, i sumy całkowitej — w jednym wyniku, żeby np. zasilić tabelę przestawną albo wydruk z podsumowaniami.

**Rozwiązanie "naiwne" — trzy osobne zapytania połączone `UNION ALL`:**

```sql
SELECT pl.Region, pl.Miasto, SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY pl.Region, pl.Miasto

UNION ALL

SELECT pl.Region, NULL AS Miasto, SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY pl.Region

UNION ALL

SELECT NULL AS Region, NULL AS Miasto, SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki;
```

Działa, ale: **trzykrotnie skanuje/agreguje te same dane** (trzy osobne zapytania), jest rozwlekły, trudny w utrzymaniu (każdy dodatkowy poziom agregacji to kolejny blok `UNION ALL`). `GROUPING SETS`/`ROLLUP`/`CUBE` robią dokładnie to samo **jednym przebiegiem przez dane, jednym zapytaniem**.

## 2. `GROUPING SETS` — jawne wypisanie potrzebnych kombinacji

```sql
SELECT pl.Region, pl.Miasto, SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY GROUPING SETS (
    (pl.Region, pl.Miasto),   -- poziom szczegółowy
    (pl.Region),              -- suma per region
    ()                        -- suma całkowita
);
```

To jest **dokładny, jednoprzebiegowy odpowiednik** wzorca `UNION ALL` z sekcji 1 — każda para nawiasów w `GROUPING SETS(...)` to jedna "kombinacja grupowania", identyczna z tym, co wcześniej pisałeś jako osobny blok `SELECT ... GROUP BY ...`. Pusta para `()` oznacza "brak grupowania" = suma po wszystkim.

**Kluczowa przewaga nad `UNION ALL`:** silnik **może** (nie zawsze musi, zależy od optymalizatora i wersji) wykonać to jednym skanem danych zamiast N osobnych — a nawet jeśli fizycznie skanuje kilka razy, kod jest zwięźlejszy i łatwiejszy w utrzymaniu: dodanie nowego poziomu agregacji to jedna linijka w `GROUPING SETS`, nie kolejny cały blok `SELECT`+`UNION ALL`.

**Pełna dowolność kombinacji** — `GROUPING SETS` nie musi tworzyć "naturalnej hierarchii" (Region→Miasto). Możesz zażądać dowolnych, niepowiązanych ze sobą zestawień w jednym zapytaniu:

```sql
GROUP BY GROUPING SETS (
    (pl.Region),
    (pl.Miasto),              -- miasto BEZ regionu — przeskakuje poziom hierarchii
    (DATEPART(YEAR, s.DataSprzedazy)),  -- zupełnie inny wymiar
    ()
);
```

## 3. `ROLLUP` — skrót dla hierarchii "od szczegółu do sumy całkowitej"

```sql
SELECT pl.Region, pl.Miasto, SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY ROLLUP (pl.Region, pl.Miasto);
```

`ROLLUP(A, B)` to skrót dokładnie równoważny:

```sql
GROUPING SETS ( (A, B), (A), () )
```

**Ważne: kolejność kolumn w `ROLLUP` ma znaczenie** — `ROLLUP` zakłada hierarchię od lewej do prawej (Region jest "nadrzędny" wobec Miasto). `ROLLUP(pl.Miasto, pl.Region)` dałoby zupełnie inne kombinacje: `(Miasto, Region), (Miasto), ()` — sumy per miasto (bez podziału na region) i sumę całkowitą, ale **nie** sumy per region. To częsty błąd — wpisanie kolumn w złej kolejności przy `ROLLUP` po cichu daje inne zestawienie, niż zamierzone.

**Kiedy używać `ROLLUP` zamiast wypisywania `GROUPING SETS` ręcznie:** gdy masz naturalną hierarchię (rok→kwartał→miesiąc, region→miasto→placówka) i chcesz podsumowań na **każdym** poziomie tej hierarchii, od najbardziej szczegółowego do sumy całkowitej. To jest krótszy zapis dokładnie tego najczęstszego przypadku użycia.

**3+ poziomy — ROLLUP skaluje się naturalnie:**

```sql
GROUP BY ROLLUP (pl.Region, pl.Miasto, pl.ID_Placowki)
-- równoważne GROUPING SETS:
-- (Region, Miasto, Placowka), (Region, Miasto), (Region), ()
```

## 4. `CUBE` — wszystkie możliwe kombinacje

```sql
SELECT pl.Region, pl.Miasto, SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY CUBE (pl.Region, pl.Miasto);
```

`CUBE(A, B)` generuje **wszystkie** kombinacje, nie tylko hierarchiczne od lewej do prawej:

```sql
GROUPING SETS ( (A, B), (A), (B), () )
```

Różnica względem `ROLLUP(A, B)`: `CUBE` dodaje **dodatkowo** kombinację `(B)` samą (Miasto bez Regionu) — sumę sprzedaży per miasto, **niezależnie od regionu**, czego `ROLLUP` w ogóle nie generuje (bo zakłada, że Miasto zawsze występuje razem z Regionem).

**Koszt rośnie wykładniczo z liczbą kolumn:** `CUBE` z N kolumnami generuje `2^N` kombinacji — `CUBE(A, B, C)` to już 8 kombinacji, `CUBE(A, B, C, D)` to 16. To jest dokładnie ten sam rodzaj ostrzeżenia, co przy `CROSSJOIN` w DAX — łatwo nieświadomie wygenerować ogromną, kosztowną liczbę grupowań, jeśli dodasz kolejną kolumnę bez zastanowienia. **Używaj `CUBE` świadomie, tylko gdy faktycznie potrzebujesz wszystkich przekrojów, nie jako domyślnego, "bezpiecznego" wyboru zamiast `ROLLUP`/`GROUPING SETS`.**

## 5. Problem — jak odróżnić prawdziwy `NULL` od `NULL` oznaczającego "suma"

To jest najważniejsza pułapka tego tematu. Spójrz na wynik `ROLLUP` z sekcji 3 — wiersz podsumowujący ma `Miasto = NULL`. **Ale co, jeśli w Twoich danych źródłowych placówka naprawdę ma `Miasto = NULL`** (np. brakujące dane, nieprzypisana placówka)? Wtedy w wyniku masz **dwa różne znaczenia tego samego `NULL`** — jedno to "suma po wszystkich miastach", drugie to "faktycznie nieznane miasto" — i nie da się ich odróżnić samym patrzeniem na kolumnę `Miasto`.

### Rozwiązanie — `GROUPING()` i `GROUPING_ID()`

```sql
SELECT
    pl.Region,
    pl.Miasto,
    SUM(s.Kwota) AS Sprzedaz,
    GROUPING(pl.Region) AS JestSumaRegionu,   -- 1 = ten wiersz to podsumowanie (Region "zwinięty"), 0 = normalny wiersz
    GROUPING(pl.Miasto) AS JestSumaMiasta,
    GROUPING_ID(pl.Region, pl.Miasto) AS PoziomAgregacji
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY ROLLUP (pl.Region, pl.Miasto);
```

**`GROUPING(kolumna)`** zwraca `1`, jeśli `NULL` w tej kolumnie pochodzi z agregacji (`ROLLUP`/`CUBE`/`GROUPING SETS` "zwinęło" tę kolumnę), i `0`, jeśli to zwykła wartość (łącznie z **prawdziwym** `NULL` z danych źródłowych — `GROUPING()` poprawnie zwróci `0` dla rzeczywistego braku danych, nie myląc go z podsumowaniem). To jest dokładnie mechanizm, który rozwiązuje niejednoznaczność z tej sekcji.

**`GROUPING_ID(kol1, kol2, ...)`** — jedna liczba całkowita kodująca kombinację wszystkich `GROUPING()` naraz (bitowo) — przydatne do szybkiego filtrowania/sortowania po "poziomie" agregacji bez pisania osobnego warunku na każdą kolumnę:

```sql
-- Pokaż tylko wiersze podsumowań per region (Miasto zwinięte, Region NIE zwinięty)
WHERE GROUPING(pl.Region) = 0 AND GROUPING(pl.Miasto) = 1

-- To samo przez GROUPING_ID (musisz znać kolejność bitów — pierwsza kolumna = bit najbardziej znaczący)
WHERE GROUPING_ID(pl.Region, pl.Miasto) = 1
```

**Praktyczne zastosowanie — czytelne etykiety zamiast `NULL` w raporcie:**

```sql
SELECT
    CASE WHEN GROUPING(pl.Region) = 1 THEN 'WSZYSTKIE REGIONY' ELSE pl.Region END AS Region,
    CASE WHEN GROUPING(pl.Miasto) = 1 THEN 'WSZYSTKIE MIASTA' ELSE ISNULL(pl.Miasto, '(brak danych)') END AS Miasto,
    SUM(s.Kwota) AS Sprzedaz
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY ROLLUP (pl.Region, pl.Miasto);
```

Zwróć uwagę na `ISNULL(pl.Miasto, '(brak danych)')` w gałęzi `ELSE` — to jest dokładnie miejsce, gdzie `GROUPING()` pozwala rozróżnić dwa znaczenia `NULL`: gdy `GROUPING(pl.Miasto) = 0`, ale sama wartość `pl.Miasto` i tak jest `NULL` (prawdziwy brak danych) — bez `GROUPING()` nie dałoby się tego bezpiecznie odróżnić od wiersza podsumowania.

## 5a. Ogólny wzorzec — kolumna z czytelnym opisem poziomu agregacji i listą kolumn zagregowanych

Sekcja 5 pokazała `GROUPING()` per kolumna. Tu budujemy to w formę gotową do bezpośredniego użycia w raporcie: jedną kolumnę tekstową wypisującą **które konkretnie kolumny są zwinięte** w danym wierszu, i drugą z czytelną nazwą poziomu — bez ręcznego wypisywania `CASE` dla każdej możliwej kombinacji.

### Lista zagregowanych kolumn — `CONCAT_WS` pomija automatycznie `NULL`

```sql
SELECT
    pl.Region,
    pl.Miasto,
    DATEPART(YEAR, s.DataSprzedazy) AS Rok,
    SUM(s.Kwota) AS Sprzedaz,
    CONCAT_WS(', ',
        CASE WHEN GROUPING(pl.Region) = 1 THEN 'Region' END,
        CASE WHEN GROUPING(pl.Miasto) = 1 THEN 'Miasto' END,
        CASE WHEN GROUPING(DATEPART(YEAR, s.DataSprzedazy)) = 1 THEN 'Rok' END
    ) AS KolumnyZagregowane
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY CUBE (pl.Region, pl.Miasto, DATEPART(YEAR, s.DataSprzedazy));
```

**Dlaczego `CONCAT_WS`, nie zwykłe `+`:** `CONCAT_WS(separator, ...)` (SQL Server 2017+) automatycznie **pomija argumenty `NULL`** i nie dokleja przy nich zbędnego separatora. Każdy `CASE WHEN GROUPING(...) = 1 THEN 'NazwaKolumny' END` zwraca albo nazwę kolumny (gdy jest zwinięta), albo `NULL` (gdy nie jest) — `CONCAT_WS` skleja tylko te niepuste, dając np. `"Region, Rok"` dla wiersza, w którym `Miasto` nie jest zagregowane. Zwykłe `+` wymagałoby ręcznego `ISNULL`/usuwania nadmiarowych przecinków.

### Czytelna nazwa poziomu — mapowanie przez `GROUPING_ID`

`GROUPING_ID(kol1, kol2, ..., kolN)` koduje kombinację jako liczbę binarną, gdzie **pierwsza kolumna na liście argumentów to bit najbardziej znaczący** (lewy), ostatnia — najmniej znaczący (prawy). Dla dwóch kolumn `GROUPING_ID(Region, Miasto)`:

| GROUPING(Region) | GROUPING(Miasto) | GROUPING_ID | Znaczenie |
|---|---|---|---|
| 0 | 0 | 0 | Szczegół — obie kolumny realne |
| 0 | 1 | 1 | Miasto zwinięte — podsumowanie per Region |
| 1 | 0 | 2 | Region zwinięty — podsumowanie per Miasto (tylko przy `CUBE`, `ROLLUP` tego nie generuje) |
| 1 | 1 | 3 | Obie zwinięte — suma całkowita |

```sql
SELECT
    pl.Region,
    pl.Miasto,
    SUM(s.Kwota) AS Sprzedaz,
    CASE GROUPING_ID(pl.Region, pl.Miasto)
        WHEN 0 THEN 'Szczegół: Region + Miasto'
        WHEN 1 THEN 'Podsumowanie: Region'
        WHEN 2 THEN 'Podsumowanie: Miasto'
        WHEN 3 THEN 'Suma całkowita'
    END AS PoziomAgregacji
FROM fact_Sprzedaz s
JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
GROUP BY CUBE (pl.Region, pl.Miasto);
```

**Zaleta względem ręcznych `CASE WHEN GROUPING(...) = 1 AND GROUPING(...) = 0 ...`:** jedna liczba całkowita zamiast łańcucha warunków `AND`/`OR` na każdej kolumnie osobno — czytelniejsze i łatwiejsze w utrzymaniu przy większej liczbie kolumn grupujących. Przy `ROLLUP` (nie `CUBE`) wartość `2` po prostu nigdy się nie pojawi w danych — `CASE` może ją pominąć albo zostawić jako nieosiągalną gałąź dla porządku.

**Ograniczenie do zapamiętania:** kolejność argumentów w `GROUPING_ID(...)` musi dokładnie odpowiadać temu, jak liczysz bity w tabeli powyżej — zmiana kolejności kolumn w `GROUPING_ID` bez odpowiedniej zmiany mapowania w `CASE` po cichu przypisze złe etykiety do wierszy. Przy więcej niż 2-3 kolumnach warto rozważyć, czy nie prościej zostać przy osobnych `GROUPING(kolumna)` per kolumna (sekcja 5) — czytelniejsze przy większej liczbie wymiarów, kosztem dłuższego kodu.

## 6. Połączenie z tematem `PIVOT` — typowy pipeline przygotowania danych

`ROLLUP`/`CUBE`/`GROUPING SETS` i `PIVOT` (z poprzedniego notatnika) często współpracują w jednym procesie ETL: `ROLLUP` buduje wszystkie potrzebne poziomy agregacji w jednym przebiegu, a `PIVOT` rozkłada wynik na szeroki format pod raport/eksport.

```sql
WITH Zagregowane AS (
    SELECT
        pl.Region,
        DATEPART(MONTH, s.DataSprzedazy) AS Miesiac,
        SUM(s.Kwota) AS Sprzedaz,
        GROUPING(pl.Region) AS JestSumaRegionu
    FROM fact_Sprzedaz s
    JOIN dim_Placowki pl ON pl.ID_Placowki = s.ID_Placowki
    GROUP BY ROLLUP (pl.Region), DATEPART(MONTH, s.DataSprzedazy)
)
SELECT
    CASE WHEN JestSumaRegionu = 1 THEN 'CAŁA FIRMA' ELSE Region END AS Region,
    [1] AS Styczen, [2] AS Luty, [3] AS Marzec
FROM Zagregowane
PIVOT ( SUM(Sprzedaz) FOR Miesiac IN ( [1], [2], [3] ) ) AS Pvt;
```

To daje w jednym, spójnym potoku danych: sumy per region **i** sumę "CAŁA FIRMA", rozłożone na kolumny miesięcy — dokładnie taki wynik, jaki często trzeba dostarczyć jako gotowy raport tabelaryczny.

## 7. Wydajność — kiedy warto, a kiedy nie

- **`GROUPING SETS`/`ROLLUP`/`CUBE` vs `UNION ALL` wielu `GROUP BY`** — teoretycznie jeden przebieg przez dane jest tańszy niż wielokrotne skanowanie, ale **optymalizator SQL Server nie zawsze faktycznie łączy to w jeden skan** — czasem generuje plan z wieloma operacjami agregacji i tak. Sprawdź `Actual Execution Plan` (dokładnie tak, jak sugerowałem przy `APPLY`/`EXISTS`) na realnych danych, zanim założysz z góry przewagę wydajnościową — ale nawet przy porównywalnym koszcie wykonania, **przewaga czytelności i łatwości utrzymania kodu pozostaje**, więc to i tak zwykle lepszy wybór niż ręczny `UNION ALL`.
- **`CUBE` na wielu kolumnach** — pamiętaj o wykładniczym wzroście liczby kombinacji (sekcja 4). Jeśli realnie potrzebujesz tylko kilku konkretnych przekrojów, a nie wszystkich możliwych, `GROUPING SETS` z jawnie wypisanymi kombinacjami jest tańsze i bardziej przewidywalne niż `CUBE` generujące wszystko, z czego część i tak odrzucisz.
- **Filtrowanie po `GROUPING()`/`GROUPING_ID()` w `WHERE` zewnętrznego zapytania** (nie w `HAVING` tego samego poziomu) — czasem konieczne opakowanie w CTE/podzapytanie, bo `GROUPING()` nie zawsze da się użyć bezpośrednio w tym samym `WHERE`, co `GROUP BY` z `ROLLUP` — sprawdź to empirycznie przy pierwszym użyciu w danym kontekście zapytania.

## 8. Podsumowanie — kiedy co

| Potrzebujesz | Rozwiązanie |
|---|---|
| Kilka **konkretnych, niehierarchicznych** kombinacji agregacji w jednym zapytaniu | `GROUPING SETS` z jawnie wypisanymi kombinacjami |
| Podsumowania na **każdym poziomie naturalnej hierarchii** (region→miasto→placówka) | `ROLLUP` — pamiętaj o kolejności kolumn (od najbardziej ogólnej) |
| **Wszystkie możliwe** kombinacje/przekroje, niezależnie od hierarchii | `CUBE` — świadomie, pamiętając o wykładniczym koszcie |
| Odróżnienie `NULL`-a oznaczającego "suma" od prawdziwego braku danych | `GROUPING(kolumna)` — zawsze, gdy kolumna może zawierać realny `NULL` |
| Filtrowanie/sortowanie po poziomie agregacji jedną wartością | `GROUPING_ID(kol1, kol2, ...)` |
| Wynik w formacie szerokim (kolumny zamiast wierszy) po agregacji wielopoziomowej | `ROLLUP`/`GROUPING SETS` w CTE, potem `PIVOT` na wyniku |